In [1]:
from typing import List, Dict
import ipywidgets as widgets
import os
import os.path as osp
from collections import OrderedDict
import json

In [2]:
import sys
sys.path.append("..")
from noisecls.data import make_dataset

In [3]:
data_dirs = [
    "/data/mid",
    "/data/processed"
]

In [4]:
class DataSelector:
    def __init__(self, data_dirs:List[str], desciption:str=None,
                 type_field=False):
        """
        data_dirs (list[str]): list of dirs with .csv files
        """
        print("Read data dirs...", flush=True)
        datas = OrderedDict()
        for data_dir in data_dirs:
            files = list(filter(lambda x: x.endswith(".csv"), os.listdir(data_dir)))
            files = sorted(files)
            print(f"{data_dir}: {len(files)} files")
            # ["name.v{X}.csv"]
            names = dict()
            for file in files:
                sep = file.index(".")
                name, vers = file[ :sep], file[sep: ]
                try:
                    names[name].append(vers)
                except KeyError:
                    names[name] = [vers]
            if names:
                datas[data_dir] = names
        self.datas = datas
        self.data_dirs = list(datas.keys())
        self.type_field = type_field
        self._init_widgets(desciption)

    def _init_widgets(self, desciption:str=None):
        layout = {'width': 'max-content'}
        self.title = widgets.Label(desciption) \
                     if desciption else None
        self.dir_select = widgets.Dropdown(options=self.data_dirs,
                                      value=None,
                                      disabled=False,
                                      layout=layout)
        self.name_select = widgets.Dropdown(disabled=True, layout=layout)
        self.vers_select = widgets.Dropdown(disabled=True, layout=layout)
        if self.type_field:
            place_hold = '/path/to/data.csv   regex are available'
            self.inp_field = widgets.Text(value=None, disabled=False,
                                          placeholder=place_hold)
        else:
            self.inp_field = widgets.Label("")
        # event trigger
        self.dir_select.observe(self._dir_change, names='value')
        self.name_select.observe(self._name_change, names='value')
        self.vers_select.observe(self._vers_change, names='value')

    def _update_path(self):
        dir_, name = self.dir_select.value, self.name_select.value
        vers = self.vers_select.value
        input_path = osp.join(dir_, name) + vers
        assert osp.exists(input_path), f"'{input_path}'"
        self.inp_field.value = input_path
        
    def _dir_change(self, event):
        data_dir = event.new
        names = list(self.datas[data_dir].keys())
        self.name_select.options = names
        self.name_select.disabled = False

    def _name_change(self, event):
        name = event.new
        data_dir = self.dir_select.value
        vers = list(self.datas[data_dir][name])
        self.vers_select.options = vers
        if len(vers) == 1:
            self.vers_select.value = vers[0]
            self.vers_select.disabled = True
            self._update_path()
        else:
            self.vers_select.options = vers
            self.vers_select.disabled = False

    def _vers_change(self, event):
        self._update_path()

    def display(self):
        hbox = widgets.HBox([self.dir_select, 
                             self.name_select, 
                             self.vers_select])
        vbox = [hbox, self.inp_field]
        if self.title:
            vbox.insert(0, self.title)
        vbox = widgets.VBox(vbox)
        return vbox

    def get_widget(self):
        return self.display()

    def get_state(self) -> str:
        return self.inp_field.value

In [5]:
class AddParams:
    def __init__(self, params:Dict[str, type]):
        wdgs = []
        self.params = dict()
        for name, t in params.items():
            if t == bool:
                wdg = widgets.Checkbox(
                    value=False,
                    description=name,
                    disabled=False,
                    indent=False
                )
            elif t == int:
                wdg = widgets.IntText(
                        description=name,
                        disabled=False,
                        layout={'width': "20ex"},
                    )
            elif t == str:
                w = "50ex" if name == "kwargs" else "28ex"
                wdg = widgets.Text(
                        disabled=False,
                        layout={'width': w},
                        description=name
            ) 
            elif isinstance(t, list):
                wdg = widgets.Dropdown(options=t,
                                       disabled=False,
                                       description=name, 
                                       layout={'width': 'max-content'})
            else:
                raise ValueError(f"Invalid param type '{t}'")
            wdgs.append(wdg)
            self.params[name] = wdg

        self.wdgs = wdgs

    def display(self):
        return widgets.VBox(self.wdgs)

    def get_widget(self):
        return self.display()

## Data transform

In [6]:
tasks = list(make_dataset.TASKS.keys())
args = {
    "input": "",
    "output": "",
    # additional kwargs
}
# name : type or 
# list for dropdown
# str for string line
# int for int line
# bool for checkbox
params = {
    "task": tasks,
    "description": str,
    "version": str,
    "tags": str,  # single str with sep ','
    "kwargs": str,  # this string will be converted to json format
                    # kwargs = '{"k1": "v1", "k2": [123, 123]}'
    "clearml": bool,
    
}

In [7]:
data_selector = DataSelector(data_dirs, desciption="Input", type_field=True)
inp_selectors = [data_selector]

out = widgets.Output()
@out.capture(clear_output=True, wait=True)
def add(b):
    data_selector_new = DataSelector(data_dirs, type_field=True)
    input_wdgs.children += (data_selector_new.get_widget(), )
    inp_selectors.append(data_selector_new)
    
button_add = widgets.Button(description="+", 
                        button_style='info'
                        ) 
button_add.on_click(add) # Назначаем этот обработчик на событие "on_click"

but_inp_ok = widgets.Button(description="Set input", 
                        button_style='info'
                        ) 
                        
input_wdgs = widgets.VBox([widgets.HBox([
    data_selector.get_widget(), 
    button_add, but_inp_ok])])

Read data dirs...
/data/mid: 27 files
/data/processed: 13 files


In [8]:
inp = args["input"].split(",")[0]
inp_dir, inp_name = osp.split(inp)
outdir_select = widgets.Dropdown(options=data_dirs,
                                 disabled=False,
                                 layout={'width': 'max-content'})
outname_print = widgets.Text(
             value=inp_name,
             disabled=False,
             layout={'width': "22ex"}
) 
but_out_ok = widgets.Button(description="Set output", 
                button_style='info' # 'success', 'info', 'warning', 'danger', ''
                ) 
out_lbl = widgets.Label("")                              

title = widgets.Label("Output:")  
out_wdgs = widgets.VBox([title, 
                         widgets.HBox([outdir_select, outname_print, but_out_ok]),
                         out_lbl])

In [ ]:
params_selector = AddParams(params)
button_run = widgets.Button(description="Run", 
                        button_style='success'
                ) 

# it will be appended iteratively
out_display = input_wdgs

def input_set(b):
    datas = [s.get_state() for s in inp_selectors]
    args["input"] = ",".join(datas)
    name = osp.split(datas[0])[1]
    outname_print.value = name
    out_display.children += (out_wdgs, )

def output_set(b):
    output = osp.join(outdir_select.value, outname_print.value)
    out_lbl.value = output
    args["output"] = output
    wdgs = widgets.VBox([
        params_selector.get_widget(),
        button_run])
    out_display.children += (wdgs, )

def run(b):
    for param, wdg in params_selector.params.items():
        args[param] = wdg.value
    kwargs : str = args.pop("kwargs")
    kwargs : dict = json.loads(kwargs) if kwargs else dict()
    args.update(kwargs)
    print("Process...")
    print(args)
    make_dataset.main(args)

but_inp_ok.on_click(input_set)
but_out_ok.on_click(output_set)
button_run.on_click(run)

display(input_wdgs)